# Chapter 2 — 데이터셋 만들기 : 코드 해설 노트북 (Python 초보용)

> 이 노트북은 `Chapter_2_Exercise_Dataset.ipynb` 의 **원본 코드 셀을 그대로 둔 채**, 번호(`# 1`, `# 2` …)가 붙은 셀마다 바로 위에 설명을 끼워 넣은 해설판입니다.
> 설명 → 코드 순서로 읽으면 됩니다. 코드 셀은 원본과 똑같으므로 그대로 실행할 수 있습니다.

## 이 챕터에서 하는 일 (한 문장)

**"글자(텍스트)를 모델이 먹을 수 있는 숫자 덩어리로 바꾸는 과정"** 을 처음부터 끝까지 만들어 봅니다.

```
 글(text)  ->  토큰 ID(정수)  ->  (입력, 정답) 쌍  ->  배치(batch)  ->  임베딩 벡터
   셀 2          셀 2~3            셀 3~4             셀 5~6           셀 7~8
```

## 먼저 알아둘 낱말 5개

| 낱말 | 쉬운 뜻 |
|---|---|
| **토큰(token)** | 글을 잘게 자른 조각. 단어 하나일 수도, 단어의 일부일 수도 있습니다. |
| **토큰 ID** | 토큰마다 붙어 있는 고유 번호(정수). 모델은 글자가 아니라 이 번호를 봅니다. |
| **토크나이저(tokenizer)** | 글 ↔ 토큰 ID 를 서로 바꿔 주는 번역기 |
| **텐서(tensor)** | PyTorch 가 쓰는 "숫자 표". 파이썬 리스트와 비슷하지만 GPU 에서 빠르게 계산됩니다. |
| **임베딩(embedding)** | 토큰 ID(정수 1개)를 의미를 담은 실수 벡터(숫자 여러 개)로 바꾼 것 |

## 셀 구성

| 셀 | 내용 |
|---|---|
| 1 | GPU 환경 설정 (모든 실습의 공통 준비) |
| 2 | 토크나이저로 글 → 숫자, 숫자 → 글 |
| 3 | 소설 한 편을 내려받아 토큰화하고, 입력 `x` 와 정답 `y` 만들어 보기 |
| 4 | `GPTDatasetV1` — (입력, 정답) 쌍을 잔뜩 만들어 보관하는 클래스 |
| 5 | `create_dataloader_v1` — 데이터를 배치로 묶어 주는 함수 |
| 6 | DataLoader 에서 실제로 배치를 꺼내 보기 |
| 7 | 임베딩 레이어 맛보기 (작은 예제) |
| 8 | 토큰 임베딩 + 위치 임베딩 = 모델 입력 완성 |
| 9 | 한국어/중국어 토크나이저 비교 (Qwen3 vs EXAONE) |

---
## 셀 1 — GPU 환경 설정

### 🎯 이 셀이 하는 일
"앞으로의 계산을 **GPU(그래픽카드)** 에서 하겠다"고 PyTorch 에게 알려 주고, 내 컴퓨터 환경 정보를 화면에 출력합니다. 딥러닝 내용 자체는 아니고 **실습 준비 단계**이므로, 처음에는 "아, 준비 운동이구나" 정도로 읽고 넘어가도 됩니다.

### 코드 뜯어보기

**① import — 도구 상자 꺼내기**
```python
import os, platform, sys, time
import torch
```
`import` 는 남이 만들어 둔 기능 묶음(모듈)을 불러오는 명령입니다. `torch` 가 바로 PyTorch 입니다.

**② 어떤 장치를 쓸지 고르기**
```python
if torch.cuda.is_available():      # NVIDIA GPU 가 있나?
    DEVICE = torch.device("cuda")
elif ...mps...:                    # 애플 실리콘(M1/M2..) GPU 가 있나?
    DEVICE = torch.device("mps")
else:                              # 둘 다 없으면
    DEVICE = torch.device("cpu")
```
`if / elif / else` 는 "만약 ~라면 / 아니고 만약 ~라면 / 그것도 아니면" 입니다. 위에서부터 차례로 검사해서 **처음 참이 되는 한 곳만** 실행합니다.
`getattr(torch.backends, "mps", None)` 은 "`torch.backends` 안에 `mps` 라는 게 있으면 꺼내고, 없으면 `None` 을 달라"는 안전한 꺼내기 방법입니다.

**③ 기본 장치 지정**
```python
torch.set_default_device(DEVICE)
```
이 한 줄 덕분에 이후에 만드는 텐서가 자동으로 GPU 에 올라갑니다. 원래는 매번 `.to(device)` 를 붙여야 하는데 그 수고를 덜어 줍니다.

**④ 시드(seed) 고정**
```python
SEED = 123
torch.manual_seed(SEED)
```
컴퓨터의 "무작위"는 사실 시드라는 출발 숫자로부터 계산됩니다. 시드를 고정하면 **실행할 때마다 같은 난수**가 나와서, 강의 자료와 내 결과를 비교하기 쉬워집니다.
그 아래 `allow_tf32`, `cudnn.benchmark` 는 GPU 계산을 조금 더 빠르게 하는 옵션입니다(몰라도 실습에 지장 없음).

**⑤ `describe_os()` 함수**
```python
def describe_os():
    ...
    return f"{system} {release}"
```
`def 이름():` 은 함수를 **정의**(만들어 두기)하는 문법이고, `return` 은 결과를 돌려주는 명령입니다. 윈도우 11 인데도 "10" 이라고 나오는 문제를 빌드 번호로 바로잡는 작은 함수입니다.

**⑥ 정보 출력 — f-string**
```python
print(f"PyTorch             : {torch.__version__}")
print("=" * 72)
```
- `f"...{값}..."` : 문자열 앞에 `f` 를 붙이면 `{ }` 안의 값이 글자로 끼워 넣어집니다.
- `"=" * 72` : 문자열에 숫자를 곱하면 그만큼 **반복**됩니다. (`=` 72개짜리 줄)
- `{값:.1f}` : 소수점 첫째 자리까지, `{값:.2f}` : 둘째 자리까지 표시하라는 뜻입니다.
- `1024**3` : `**` 는 거듭제곱. 바이트를 GB 로 바꾸려고 1024 를 세 번 곱한 값으로 나눕니다.

**⑦ 속도 측정**
```python
a, b = torch.randn(2048, 2048), torch.randn(2048, 2048)
t0 = time.perf_counter()
_ = a @ b
print(... (time.perf_counter() - t0) * 1000 ...)
```
- `torch.randn(2048, 2048)` : 무작위 숫자로 채운 2048×2048 표를 만듭니다.
- `a @ b` : `@` 는 **행렬 곱** 연산자입니다. (앞으로 정말 많이 나옵니다)
- `_ = ...` : 결과를 쓰지 않을 때 관례적으로 `_` 라는 이름에 버립니다.
- 끝난 시각 − 시작 시각 = 걸린 시간. 스톱워치와 같습니다.
- `del a, b, probe` : 다 쓴 변수를 지워 메모리를 돌려줍니다.

### ⚠️ 주석에 적힌 주의사항 풀이
기본 장치를 GPU 로 바꿔 두면 편하지만, 그 대신 ① `DataLoader` 는 `num_workers=0` 으로 써야 하고 ② 텐서를 넘파이로 바꿀 때는 `tensor.cpu().numpy()` 처럼 **먼저 CPU 로 내려야** 합니다.

### 실행 결과 읽기
출력에 `CUDA 사용 가능 : True`, `검증용 텐서 생성 위치 : cuda:0` 이 보이면 GPU 설정이 잘 된 것입니다.

In [1]:
# 1
# ============================================================================
# [공통 환경 설정] 모든 실습을 CPU가 아닌 GPU에서 구동하기 위한 설정
#  - 이 셀을 가장 먼저 실행하면, 이후 만들어지는 텐서/모델이 기본적으로 GPU에 올라갑니다.
#  - GPU가 없는 환경에서는 자동으로 CPU로 대체되고 경고 메시지를 출력합니다.
# ============================================================================
import os
import platform
import sys
import time

import torch

# ---------------------------------------------------------------------------
# 1. 사용할 장치(device) 결정: CUDA(NVIDIA) > MPS(Apple Silicon) > CPU 순
# ---------------------------------------------------------------------------
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

device = DEVICE  # 다른 챕터들이 사용하는 소문자 이름도 함께 정의

# ---------------------------------------------------------------------------
# 2. 앞으로 생성되는 모든 텐서/레이어가 기본적으로 GPU에 만들어지도록 지정
#    -> .to(device) 를 깜빡해도 GPU에서 동작합니다.
#    [주의] 이 설정에서는 DataLoader를 num_workers=0, pin_memory=False 로 두세요.
#           numpy로 바꿀 때는 tensor.cpu().numpy() 를 사용해야 합니다.
# ---------------------------------------------------------------------------
torch.set_default_device(DEVICE)

# ---------------------------------------------------------------------------
# 3. GPU 연산 가속 옵션 + 재현성을 위한 시드 고정
# ---------------------------------------------------------------------------
SEED = 123
torch.manual_seed(SEED)

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True  # 행렬 곱에 TF32 사용 (속도 향상)
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True         # 입력 크기가 일정하면 최적 커널을 자동 선택
    torch.cuda.empty_cache()

# ---------------------------------------------------------------------------
# 4. 환경 정보 출력
# ---------------------------------------------------------------------------
def describe_os():
    """Windows 11도 platform.release()가 '10'을 반환하므로 빌드 번호로 보정합니다."""
    system, release = platform.system(), platform.release()
    if system == "Windows":
        build = platform.version().split(".")[-1]  # 예: '10.0.26200' -> '26200'
        if build.isdigit():
            release = "11" if int(build) >= 22000 else release  # 빌드 22000 이상 = Windows 11
            return f"{system} {release} (build {build})"
    return f"{system} {release}"


print("=" * 72)
print("공통 환경 설정 (GPU)")
print("=" * 72)
print(f"OS                  : {describe_os()}")
print(f"Python              : {sys.version.split()[0]}")
print(f"PyTorch             : {torch.__version__}")
print(f"CUDA (빌드 버전)    : {torch.version.cuda}")
print(f"cuDNN               : {torch.backends.cudnn.version()}")
print(f"CUDA 사용 가능      : {torch.cuda.is_available()}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', '(미설정 → 전체 GPU 사용)')}")
print("-" * 72)

if DEVICE.type == "cuda":
    print(f"인식된 GPU 개수     : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        print(f"  [GPU {i}] {prop.name}")
        print(f"          - Compute Capability : {prop.major}.{prop.minor}")
        print(f"          - 전체 메모리(VRAM)  : {prop.total_memory / 1024**3:.1f} GB")
        print(f"          - SM(멀티프로세서) 수: {prop.multi_processor_count}")
    current = torch.cuda.current_device()
    print(f"현재 사용 GPU       : cuda:{current} ({torch.cuda.get_device_name(current)})")
    print(f"메모리 사용량       : 할당 {torch.cuda.memory_allocated() / 1024**2:.1f} MB / "
          f"예약 {torch.cuda.memory_reserved() / 1024**2:.1f} MB")
    print(f"TF32 / cuDNN benchmark: {torch.backends.cuda.matmul.allow_tf32} / "
          f"{torch.backends.cudnn.benchmark}")
else:
    print("[경고] 사용 가능한 GPU가 없어 CPU로 실행됩니다. 학습 속도가 매우 느릴 수 있습니다.")
    print("       - NVIDIA 드라이버와 CUDA 지원 PyTorch가 설치되어 있는지 확인하세요.")
    print("       - 설치 예시: pip install torch --index-url https://download.pytorch.org/whl/cu128")

print("-" * 72)
print(f"기본 장치(default device) : {DEVICE}")
print(f"기본 dtype                : {torch.get_default_dtype()}")
print(f"고정된 시드(SEED)         : {SEED}")

# 실제로 GPU에 텐서가 만들어지는지 확인 + 행렬 곱 속도 측정 (2048 x 2048)
probe = torch.zeros(1)
print(f"검증용 텐서 생성 위치     : {probe.device}")

a, b = torch.randn(2048, 2048), torch.randn(2048, 2048)
_ = a @ b                                  # 워밍업 (첫 호출은 커널 로딩 시간 포함)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
_ = a @ b
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
print(f"2048x2048 행렬 곱 소요    : {(time.perf_counter() - t0) * 1000:.2f} ms ({DEVICE.type})")

del a, b, probe
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print("=" * 72)

공통 환경 설정 (GPU)
OS                  : Windows 11 (build 26300)
Python              : 3.13.17
PyTorch             : 2.11.0+cu128
CUDA (빌드 버전)    : 12.8
cuDNN               : 91900
CUDA 사용 가능      : True
CUDA_VISIBLE_DEVICES: (미설정 → 전체 GPU 사용)
------------------------------------------------------------------------
인식된 GPU 개수     : 1
  [GPU 0] NVIDIA GeForce RTX 5070 Laptop GPU
          - Compute Capability : 12.0
          - 전체 메모리(VRAM)  : 8.0 GB
          - SM(멀티프로세서) 수: 36
현재 사용 GPU       : cuda:0 (NVIDIA GeForce RTX 5070 Laptop GPU)
메모리 사용량       : 할당 0.0 MB / 예약 0.0 MB
TF32 / cuDNN benchmark: True / True
------------------------------------------------------------------------
기본 장치(default device) : cuda
기본 dtype                : torch.float32
고정된 시드(SEED)         : 123
검증용 텐서 생성 위치     : cuda:0
2048x2048 행렬 곱 소요    : 1.78 ms (cuda)


---
## 셀 2 — 토크나이저 : 글 ↔ 숫자

### 🎯 이 셀이 하는 일
문장을 **토큰 ID(정수 리스트)** 로 바꾸고(인코딩), 그 숫자를 다시 문장으로 되돌립니다(디코딩).

```
 "Hello, do you like tea?"  --encode-->  [15496, 11, 466, 345, 588, 8887, 30]
 [15496, 11, 466, ...]      --decode-->  "Hello, do you like tea?"
```

### 코드 뜯어보기

| 코드 | 설명 |
|---|---|
| `import tiktoken` | OpenAI 가 만든 토크나이저 라이브러리를 불러옵니다. |
| `importlib.metadata.version("tiktoken")` | 설치된 tiktoken 의 버전을 알려 줍니다. |
| `tokenizer = tiktoken.get_encoding("gpt2")` | GPT-2 가 쓰던 단어장(약 5만 개 토큰)을 가진 토크나이저를 준비합니다. |
| `tokenizer.encode(text, ...)` | **글 → 숫자 리스트** |
| `tokenizer.decode(integers)` | **숫자 리스트 → 글** |

**`text = ( "..." "..." )` 는 뭘까?**
```python
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
```
파이썬은 따옴표 문자열 두 개를 **나란히 쓰면 자동으로 이어 붙입니다.** 긴 문장을 여러 줄에 나눠 쓰려고 괄호로 감싼 것입니다.
그런데 첫 줄 끝과 둘째 줄 앞에 띄어쓰기가 없어서 실제 문장은 `...terracesof some...` 이 됩니다. 출력에서 `terracesof` 로 붙어 나오는 이유가 이것입니다. (원서 예제가 원래 이렇습니다. "처음 보는 이상한 단어도 토크나이저가 처리할 수 있다"를 보여주는 데는 오히려 도움이 됩니다.)

**`allowed_special={"<|endoftext|>"}`**
`<|endoftext|>` 는 "여기서 글 하나가 끝났다"를 뜻하는 **특수 토큰**입니다. 이 옵션은 "이 특수 표시를 일반 글자로 쪼개지 말고 토큰 1개(ID 50256)로 인정해 달라"는 뜻입니다. `{ }` 는 **집합(set)** 자료형입니다.

### 💡 `# region` / `# endregion` / `# TODO` 는?
- `# region ... # endregion` : 에디터(VS Code)에서 코드를 접었다 폈다 하기 위한 표시일 뿐, 실행에는 아무 영향이 없습니다.
- `# TODO` 와 `????` 가 들어간 주석 줄 : 원래 **빈칸 채우기 문제**였던 자리입니다. 바로 아랫줄이 정답이 채워진 실제 코드입니다. 이 노트북 전체가 같은 형식입니다.

### 실행 결과 읽기
```
[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
```
- `15496` = "Hello", `11` = ",", `50256` = `<|endoftext|>` 입니다.
- `someunknownPlace` 처럼 사전에 없는 단어는 `some` + `unknown` + `Place` 처럼 **아는 조각들로 쪼개집니다.** 이 방식을 BPE(Byte Pair Encoding)라고 하며, 덕분에 "모르는 단어"가 생기지 않습니다.
- 마지막 줄은 디코딩 결과로, 원래 문장이 그대로 복원됩니다.

In [2]:
# 2
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
# region  [token encoding]
# TODO: 인코딩 메서드명을 채워 text를 토큰 ID로 변환하세요.
# 힌트: `????`에는 문자열을 토큰 ID 리스트로 바꾸는 인코딩 메서드 이름이 들어갑니다.
# 힌트: `tokenizer.` 뒤 메서드 이름만 채우고 `allowed_special` 설정은 그대로 두면 됩니다.
# integers = tokenizer.????(text, allowed_special={"<|endoftext|>"})
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
# endregion
print(integers)

# region  [token decoding]
# TODO: 디코딩 메서드명을 채워 토큰 ID를 문자열로 복원하세요.
# 힌트: `????`에는 토큰 ID 리스트를 다시 문자열로 바꾸는 디코딩 메서드 이름이 들어갑니다.
# 힌트: `tokenizer.` 뒤 메서드 이름만 채우고 입력은 `integers` 그대로 사용하세요.
# strings = tokenizer.????(integers)
strings = tokenizer.decode(integers)
# endregion
print(strings)


tiktoken version: 0.14.0
[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


---
## 셀 3 — 소설 내려받기 + 입력 `x` 와 정답 `y` 만들기

### 🎯 이 셀이 하는 일
1. 학습에 쓸 짧은 소설(`the-verdict.txt`)을 인터넷에서 내려받아 저장하고
2. 통째로 토큰화한 뒤
3. GPT 학습의 핵심 아이디어인 **"정답은 입력을 한 칸 민 것"** 을 눈으로 확인합니다.

### 코드 뜯어보기 — ① 파일 준비

```python
os.makedirs("data", exist_ok=True)
```
`data` 폴더를 만듭니다. `exist_ok=True` 는 "이미 있어도 에러 내지 마"입니다.

```python
if not os.path.exists("data/the-verdict.txt"):
    ...
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)
```
- `if not os.path.exists(...)` : 파일이 **없을 때만** 내려받습니다. (두 번째 실행부터는 건너뜀)
- `requests.get(url, timeout=30)` : 그 주소의 내용을 가져옵니다. 30초 넘게 응답이 없으면 포기합니다.
- `raise_for_status()` : 내려받기가 실패(404 등)했으면 여기서 에러를 내서 알려 줍니다.
- `with open(경로, "wb") as f:` : 파일을 열고, 블록이 끝나면 **자동으로 닫아** 줍니다. `"wb"` 는 write(쓰기) + binary(원본 바이트 그대로).

```python
with open("data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
```
이번에는 `"r"`(read, 읽기)로 열어 파일 전체를 문자열 하나(`raw_text`)로 읽습니다.

### 코드 뜯어보기 — ② 토큰화와 슬라이싱

```python
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))        # 5145
```
소설 전체가 **5145개의 토큰**으로 바뀌었습니다. `len()` 은 리스트의 길이를 셉니다.

```python
enc_sample = enc_text[50:]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1: context_size + 1]
```

**슬라이싱(slicing)** : `리스트[시작:끝]` 은 시작 번호부터 **끝 번호 바로 앞까지** 잘라 냅니다. (번호는 0부터 셉니다)

| 코드 | 뜻 |
|---|---|
| `enc_text[50:]` | 50번부터 끝까지 (앞의 50개는 버림) |
| `enc_sample[:4]` | 처음부터 3번까지 → 0,1,2,3번 (4개) |
| `enc_sample[1:5]` | 1번부터 4번까지 → 1,2,3,4번 (4개) |

### ⭐ 핵심 : 왜 y 는 한 칸 밀려 있을까?

GPT 는 **"지금까지의 글을 보고 다음 토큰을 맞히는"** 모델입니다. 그래서 정답지는 입력을 오른쪽으로 한 칸 밀기만 하면 저절로 만들어집니다.

```
 x: [290, 4920, 2241, 287]
 y:      [4920, 2241, 287, 257]

 290                   을 보면 -> 4920 을 맞혀야 함
 290 4920              을 보면 -> 2241 을 맞혀야 함
 290 4920 2241         을 보면 -> 287  을 맞혀야 함
 290 4920 2241 287     을 보면 -> 257  을 맞혀야 함
```
사람이 정답을 따로 달아 줄 필요가 없다는 점이 중요합니다. 글 자체가 문제집이자 정답지입니다.

`context_size`(문맥 크기) 는 "모델이 한 번에 볼 토큰 수"입니다. 여기서는 이해를 위해 4로 아주 작게 잡았습니다.

In [3]:
# 3
import os
import requests

os.makedirs("data", exist_ok=True)
if not os.path.exists("data/the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "data/the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)
with open("data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

enc_sample = enc_text[50:]
context_size = 4

# TODO: y 시퀀스가 x보다 한 칸 오른쪽으로 미루도록 인덱스를 채우세요.
# 힌트: 첫 번째 `????`에는 시작 인덱스를 한 칸 미루는 값 `1`이 들어갑니다.
# 힌트: 두 번째 `?????`에도 끝 인덱스를 한 칸 늘리는 값 `1`이 들어가며, y는 x의 next-token 시퀀스가 됩니다.
x = enc_sample[:context_size]
# y = enc_sample[????:context_size+?????]
y = enc_sample[1: context_size + 1]

print(f"x: {x}")
print(f"y:      {y}")

5145
x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


---
## 셀 4 — `GPTDatasetV1` 클래스 (⭐ 중요)

### 🎯 이 셀이 하는 일
셀 3에서 손으로 한 번 해 본 "x, y 만들기"를 **소설 전체에 대해 자동으로 반복**해서, (입력, 정답) 쌍을 잔뜩 만들어 보관하는 **데이터셋 클래스**를 정의합니다.

### 먼저 : 클래스(class)란?
클래스는 **"데이터 + 그 데이터를 다루는 함수"를 한 묶음으로 만든 설계도**입니다. 붕어빵 틀(클래스)로 붕어빵(객체)을 찍어낸다고 생각하면 됩니다.

```python
class GPTDatasetV1(Dataset):
```
괄호 안의 `Dataset` 은 **상속**입니다. "PyTorch 가 정해 둔 `Dataset` 규격을 따르는 클래스를 만들겠다"는 뜻입니다. 이 규격을 따르려면 아래 세 가지 **특별한 메서드**를 꼭 만들어야 합니다.

| 메서드 | 언제 자동으로 불리나 | 역할 |
|---|---|---|
| `__init__` | `GPTDatasetV1(...)` 로 객체를 만들 때 | 초기 준비 (여기서 데이터를 전부 만들어 둠) |
| `__len__` | `len(dataset)` 할 때 | 샘플이 몇 개인지 알려 줌 |
| `__getitem__` | `dataset[3]` 처럼 꺼낼 때 | 그 번호의 (입력, 정답) 쌍을 돌려줌 |

이름 앞뒤에 밑줄 두 개(`__`)가 붙은 메서드는 파이썬이 **특정 상황에서 알아서 호출**해 주는 약속된 이름입니다.

`self` 는 "나 자신(지금 만들어진 이 객체)"을 가리킵니다. `self.input_ids = []` 는 "내 안에 `input_ids` 라는 빈 리스트 보관함을 만든다"는 뜻입니다.

### `__init__` 뜯어보기

**① 전체 글을 토큰화**
```python
token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
```

**② 길이 검사**
```python
assert len(token_ids) > max_length, "..."
```
`assert 조건, "메시지"` 는 **조건이 거짓이면 즉시 에러**를 내는 안전장치입니다. 글이 너무 짧으면 한 쌍도 못 만들기 때문입니다.

**③ 슬라이딩 윈도우 (창문을 옆으로 밀면서 자르기)**
```python
for i in range(0, len(token_ids) - max_length, stride):
    input_chunk  = token_ids[i     : i + max_length]
    target_chunk = token_ids[i + 1 : i + max_length + 1]
    self.input_ids.append(torch.tensor(input_chunk))
    self.target_ids.append(torch.tensor(target_chunk))
```
- `range(시작, 끝, 간격)` : 시작부터 끝 **직전**까지 간격만큼 건너뛰며 숫자를 만듭니다. 예) `range(0, 10, 4)` → 0, 4, 8
- `input_chunk` : i 번부터 `max_length` 개
- `target_chunk` : **한 칸 뒤**(i+1 번)부터 `max_length` 개 → 셀 3의 x, y 와 똑같은 원리
- `torch.tensor(리스트)` : 파이썬 리스트를 텐서로 변환
- `.append(...)` : 리스트 맨 뒤에 하나 추가

`max_length=4`, `stride=2` 일 때의 그림입니다.

```
 tokens :  t0 t1 t2 t3 t4 t5 t6 t7 t8 ...

 i=0  input  [t0 t1 t2 t3]
      target    [t1 t2 t3 t4]
 i=2  input        [t2 t3 t4 t5]
      target          [t3 t4 t5 t6]
 i=4  input              [t4 t5 t6 t7]
      target                [t5 t6 t7 t8]
```

**`stride`(보폭) 의 의미**
- `stride=1` : 한 칸씩 이동 → 샘플이 아주 많지만 서로 많이 겹침
- `stride=max_length` : 창 크기만큼 이동 → 겹침 없이 깔끔하게 나눔

**왜 `len(token_ids) - max_length` 에서 멈출까?** target 은 input 보다 한 칸 더 뒤까지 필요합니다. 끝까지 가 버리면 정답으로 쓸 토큰이 모자라므로 그만큼 일찍 멈춥니다.

### `__len__` 과 `__getitem__`
```python
def __len__(self):
    return len(self.input_ids)

def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]
```
- `__len__` : 만들어 둔 샘플 개수를 돌려줍니다.
- `__getitem__` : `idx` 번째 입력과 정답을 **쌍으로** 돌려줍니다. `return a, b` 처럼 쉼표로 두 개를 돌려주면 **튜플**(묶음) `(a, b)` 가 됩니다. 반드시 **같은 번호**끼리 짝지어야 문제와 정답이 어긋나지 않습니다.

### 💡 `"""..."""` 는?
함수/클래스 바로 아래 큰따옴표 세 개로 감싼 글은 **독스트링(docstring)**, 즉 사용 설명서입니다. 실행에는 영향이 없습니다.

> 이 셀은 클래스를 **정의만** 하므로 실행해도 출력이 없습니다. 실제로 쓰는 것은 셀 5, 6 입니다.

In [4]:
# 4
### 중요: 이 코드는 GPT 모델 학습을 위한 데이터셋 클래스를 정의합니다

import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        """
        Args:
            txt (str): 학습할 전체 텍스트 데이터
            tokenizer: 텍스트를 토큰 ID로 변환해주는 토크나이저 (예: tiktoken)
            max_length (int): 모델이 한 번에 볼 수 있는 윈도우 크기 (입력 시퀀스 길이)
            stride (int): 윈도우를 이동시킬 간격 (데이터 중복 정도를 결정)
        """
        self.input_ids = []
        self.target_ids = []

        # 1. 전체 텍스트 토큰화
        # 텍스트를 정수 리스트(token_ids)로 변환합니다.
        # <|endoftext|> 같은 특수 토큰도 허용하여 인코딩합니다.
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # 데이터가 너무 짧으면 학습할 수 없으므로 최소 길이를 확인합니다.
        assert len(token_ids) > max_length, "토큰화된 입력의 개수는 적어도 max_length+1과 같아야 합니다."

        # 2. 슬라이딩 윈도우(Sliding Window)로 데이터 생성
        # 전체 토큰 리스트를 훑으며 max_length 길이만큼 잘라냅니다.
        # stride만큼 건너뛰며 반복합니다.
        for i in range(0, len(token_ids) - max_length, stride):
            # 입력 청크: 현재 위치(i)부터 max_length만큼 가져옵니다.
            # region [input 청크 생성]
            # TODO: 입력 청크 슬라이싱 구간을 채우세요.
            # 힌트: 첫 번째 `????`는 슬라이싱 시작 위치인 `i`입니다.
            # 힌트: 두 번째 `????`도 끝 위치 계산에 쓰이는 `i`라서 `i : i + max_length` 형태를 완성하면 됩니다.
            # input_chunk = token_ids[???? : ???? + max_length]
            input_chunk = token_ids[i : i + max_length]
            # endregion
            
            # 타겟 청크: 입력보다 1칸 뒤의 위치(i+1)부터 가져옵니다.
            # GPT는 '다음 단어'를 맞추는 모델이므로, 정답은 입력보다 한 칸씩 뒤로 밀려있어야 합니다.
            # region [target 청크 생성]
            # TODO: 타깃 청크 슬라이싱 구간을 채우세요.
            # 힌트: 네 개의 `????`는 순서대로 `i`, `1`, `i`, `1`입니다.
            # 힌트: 입력보다 한 칸 뒤를 가리키도록 `i + 1 : i + max_length + 1` 형태를 만들면 됩니다.
            # target_chunk = token_ids[???? + ???? : ???? + max_length + ????]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            # endregion

            # 추출한 데이터를 텐서(Tensor)로 변환하여 리스트에 저장합니다.
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # 데이터셋의 총 샘플(청크) 개수를 반환합니다.
        return len(self.input_ids)

    def __getitem__(self, idx):
        # DataLoader가 데이터를 요청할 때 호출됩니다.
        # 해당 인덱스(idx)의 입력과 정답 쌍을 반환합니다.
        # TODO: __getitem__에서 같은 idx의 입력/타깃 쌍을 반환하세요.
        # 힌트: `self.input_ids`와 `self.target_ids`에서 `idx` 위치의 값을 각각 반환하세요.
        # return self.input_ids[????], self.target_ids[????]
        return self.input_ids[idx], self.target_ids[idx]

---
## 셀 5 — `create_dataloader_v1` 함수 (⭐ 중요)

### 🎯 이 셀이 하는 일
"글만 넣으면 → 토크나이저 준비 → Dataset 생성 → DataLoader 생성"까지 한 번에 해 주는 **도우미(헬퍼) 함수**를 만듭니다.

### Dataset 과 DataLoader 의 차이

| | 비유 | 역할 |
|---|---|---|
| **Dataset** | 문제집 | 샘플을 **한 개씩** 보관하고 꺼내 줌 |
| **DataLoader** | 문제집에서 몇 장씩 뜯어 나눠 주는 조교 | 샘플을 **여러 개씩 묶어(batch)** 차례로 넘겨 줌 |

모델은 한 번에 한 문제씩 푸는 것보다 여러 문제를 묶어서 푸는 편이 훨씬 빠르기 때문에 DataLoader 가 필요합니다.

### 코드 뜯어보기

```python
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
```
`batch_size=4` 처럼 `=값` 이 붙은 것은 **기본값**입니다. 함수를 부를 때 그 인자를 생략하면 기본값이 쓰입니다.

| 인자 | 뜻 |
|---|---|
| `txt` | 학습할 글 전체 |
| `batch_size` | 한 번에 묶을 샘플 개수 |
| `max_length` | 샘플 하나의 토큰 수 (창문 크기) |
| `stride` | 창문을 미는 간격 |
| `shuffle` | `True` 면 매번 샘플 순서를 섞음 (학습용은 보통 섞음) |
| `drop_last` | `True` 면 마지막에 남는 자투리 배치(개수가 모자란 묶음)를 버림 |
| `num_workers` | 데이터를 준비하는 보조 프로세스 수. 0 이면 혼자 처리 (셀 1 주의사항대로 0 유지) |

```python
tokenizer = tiktoken.get_encoding("gpt2")
dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                        drop_last=drop_last, num_workers=num_workers)
return dataloader
```
- `GPTDatasetV1(...)` : 셀 4의 설계도로 실제 객체를 만듭니다. 이 순간 `__init__` 이 실행되어 모든 (입력, 정답) 쌍이 만들어집니다.
- `DataLoader(dataset, ...)` : **첫 번째 인자가 데이터셋**입니다. (빈칸 문제의 정답)
- `batch_size=batch_size` : 왼쪽은 DataLoader 의 인자 이름, 오른쪽은 우리 함수가 받은 값입니다. 이름이 같아서 헷갈리지만 "받은 값을 그대로 전달한다"는 뜻입니다.

> 이 셀도 함수 **정의만** 하므로 출력이 없습니다.

In [5]:
# 5
##중요 : 아래 함수는 DataLoader를 생성하는 헬퍼 함수입니다.
import tiktoken
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # 토크나이저를 초기화합니다.
    tokenizer = tiktoken.get_encoding("gpt2")

    # 데이터셋을 만듭니다.
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 데이터 로더를 만듭니다.
    # region [DataLoader 생성]
    # TODO: DataLoader의 첫 번째 인자를 채우세요.
    # 힌트: `????`에는 바로 위에서 생성한 `dataset` 객체가 들어갑니다.
    # 힌트: DataLoader의 첫 번째 위치 인자는 순회할 데이터셋 자체입니다.
    dataloader = DataLoader(
        # ????,
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    # endregion    
    return dataloader

---
## 셀 6 — DataLoader 에서 배치 꺼내 보기

### 🎯 이 셀이 하는 일
셀 5의 함수로 DataLoader 를 만들고, 배치를 두 번 꺼내서 **정말로 한 칸씩 밀린 쌍이 나오는지** 확인합니다.

### 코드 뜯어보기
```python
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)
```
관찰하기 쉽게 일부러 작게 잡았습니다. 한 번에 1개씩(`batch_size=1`), 토큰 4개짜리(`max_length=4`), 한 칸씩 이동(`stride=1`), 섞지 않음(`shuffle=False`).

```python
data_iter = iter(dataloader)
first_batch = next(data_iter)
second_batch = next(data_iter)
```
- `iter(...)` : "차례대로 꺼낼 준비"를 합니다 (번호표 기계를 켜는 것).
- `next(...)` : **다음 것 하나**를 꺼냅니다. 부를 때마다 그다음 배치가 나옵니다.

보통은 `for batch in dataloader:` 로 한꺼번에 돌지만, 여기서는 하나씩 들여다보려고 `iter` / `next` 를 썼습니다.

### 실행 결과 읽기
```
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]
```
- 각 줄은 `[입력 텐서, 정답 텐서]` 입니다.
- 첫 줄 : 입력 `40, 367, 2885, 1464` → 정답 `367, 2885, 1464, 1807` (한 칸 밀림 ✔)
- 둘째 줄의 입력은 첫 줄 입력에서 **한 칸 이동**했습니다 (`stride=1` ✔)
- 대괄호가 두 겹 `[[ ... ]]` 인 이유 : 모양이 `[배치 크기 1, 토큰 4개]` 인 2차원 텐서이기 때문입니다.
- `device='cuda:0'` : 셀 1 설정대로 GPU 에 만들어졌다는 표시입니다.

**직접 해 보기** : `stride=4` 나 `batch_size=2` 로 바꿔 실행하면 어떻게 달라지는지 확인해 보세요.

In [6]:
# 6
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)

first_batch = next(data_iter)
print(first_batch)

second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464]], device='cuda:0'), tensor([[ 367, 2885, 1464, 1807]], device='cuda:0')]
[tensor([[ 367, 2885, 1464, 1807]], device='cuda:0'), tensor([[2885, 1464, 1807, 3619]], device='cuda:0')]


---
## 셀 7 — 임베딩 레이어 맛보기

### 🎯 이 셀이 하는 일
토큰 ID(정수)를 **벡터(실수 여러 개)** 로 바꿔 주는 임베딩 레이어를 아주 작은 크기로 만들어 동작을 확인합니다.

### 왜 임베딩이 필요할까?
토큰 ID 는 그냥 **번호표**입니다. 5번 토큰이 2번 토큰보다 "크다"거나 "2.5배"라는 의미는 전혀 없습니다. 신경망은 숫자의 크기로 계산을 하기 때문에, 번호표 대신 **의미를 담을 수 있는 숫자 묶음(벡터)** 으로 바꿔 줘야 합니다.

### 코드 뜯어보기
```python
input_ids = torch.tensor([2, 3, 5, 1])   # 토큰 ID 4개
vocab_size = 6                           # 단어장 크기 : 토큰이 6종류(0~5번)뿐인 장난감 단어장
output_dim = 3                           # 토큰 하나를 숫자 3개로 표현
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
```
`torch.nn.Embedding(6, 3)` 은 **6행 3열짜리 숫자 표**를 만듭니다. 처음에는 무작위 숫자로 채워집니다.

```python
print(embedding_layer.weight)
```
`.weight` 가 바로 그 표(가중치)입니다.

```python
token_embeddings = embedding_layer(input_ids)
```
레이어에 ID 를 넣으면 **그 번호에 해당하는 행을 꺼내 줍니다.** 계산이라기보다 "표에서 찾아오기(look-up)"입니다.

### 실행 결과 읽기
```
 weight (6 x 3)                         input_ids = [2, 3, 5, 1]
 row0 [ 1.3391,  0.2052, -1.6879]
 row1 [ 0.5103, -0.3458,  0.6455]  <--- 4th (id 1)
 row2 [ 1.5735,  0.3519,  1.1298]  <--- 1st (id 2)
 row3 [ 0.0098, -0.3148,  0.4809]  <--- 2nd (id 3)
 row4 [-1.2309,  1.0901,  1.0566]
 row5 [-0.9747, -0.3215, -1.6100]  <--- 3rd (id 5)
```
두 번째 출력(`token_embeddings`)이 정확히 2번, 3번, 5번, 1번 행을 순서대로 모아 놓은 것임을 직접 대조해 보세요.

- `requires_grad=True` : "이 숫자들은 **학습으로 바뀔 수 있다**"는 표시입니다. 지금은 무작위지만, 학습이 진행되면 비슷한 뜻의 토큰끼리 비슷한 벡터를 갖도록 조정됩니다.
- `torch.manual_seed(123)` 을 바로 위에서 다시 부른 것은 무작위 표가 강의 자료와 똑같이 나오게 하기 위해서입니다.

In [7]:
# 7
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(123)
# region [임베딩 레이어 생성]
# TODO: Embedding 생성자 인자를 채우세요.
# 힌트: 첫 번째 `????`는 단어장 크기인 `vocab_size`입니다.
# 힌트: 두 번째 `????`는 각 토큰 임베딩 벡터의 차원인 `output_dim`입니다.
# embedding_layer = torch.nn.Embedding(????, ????)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
# endregion

print(embedding_layer.weight)

# region [임베딩 레이어 적용]
# TODO: 임베딩 레이어 입력 변수를 채우세요.
# 힌트: `????`에는 임베딩할 토큰 ID 텐서인 `input_ids`가 들어갑니다.
# 힌트: Embedding 레이어는 정수 토큰 ID 텐서를 입력으로 받습니다.
# token_embeddings = embedding_layer(????)
token_embeddings = embedding_layer(input_ids)
# endregion
print(token_embeddings)


Parameter containing:
tensor([[ 1.3391,  0.2052, -1.6879],
        [ 0.5103, -0.3458,  0.6455],
        [ 1.5735,  0.3519,  1.1298],
        [ 0.0098, -0.3148,  0.4809],
        [-1.2309,  1.0901,  1.0566],
        [-0.9747, -0.3215, -1.6100]], device='cuda:0', requires_grad=True)
tensor([[ 1.5735,  0.3519,  1.1298],
        [ 0.0098, -0.3148,  0.4809],
        [-0.9747, -0.3215, -1.6100],
        [ 0.5103, -0.3458,  0.6455]], device='cuda:0',
       grad_fn=<EmbeddingBackward0>)


---
## 셀 8 — 토큰 임베딩 + 위치 임베딩

### 🎯 이 셀이 하는 일
실제 GPT-2 크기의 단어장(50257개)으로 임베딩을 만들고, 여기에 **"몇 번째 자리인지"** 정보를 더해 모델에 들어갈 최종 입력을 완성합니다.

### 왜 위치 정보가 필요할까?
임베딩은 "표에서 찾아오기"라서, 같은 토큰은 문장 **어디에 있든 똑같은 벡터**가 됩니다. 그러면 "개가 사람을 물었다"와 "사람이 개를 물었다"를 구분하기 어렵습니다. 그래서 0번 자리, 1번 자리, 2번 자리… 마다 고유한 벡터(**위치 임베딩**)를 따로 만들어 **더해** 줍니다.

### 코드 뜯어보기

**① 토큰 임베딩**
```python
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
```
50257행 × 256열의 표입니다. 토큰 하나가 숫자 256개로 표현됩니다.

**② 배치 하나 꺼내기**
```python
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length,
                                  stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
```
`inputs, targets = ...` 는 **언패킹**입니다. 돌아온 `[입력, 정답]` 묶음을 두 변수에 나눠 담습니다. `inputs` 의 모양은 `[8, 4]` (샘플 8개 × 토큰 4개).

**③ 임베딩 적용**
```python
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)      # torch.Size([8, 4, 256])
```
`.shape` 는 텐서의 **모양**(각 방향의 크기)입니다. 토큰 하나하나가 256개 숫자로 바뀌어서 `[8, 4]` → `[8, 4, 256]` 이 됐습니다.

**④ 위치 임베딩**
```python
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)        # torch.Size([4, 256])
```
- 자리가 4개뿐이므로 4행 × 256열 표를 만듭니다.
- `torch.arange(4)` 는 `[0, 1, 2, 3]` 을 만듭니다(= 자리 번호). 이걸 넣어 0~3번 자리의 벡터를 모두 꺼냅니다.

**⑤ 더하기**
```python
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)      # torch.Size([8, 4, 256])
```
모양이 `[8, 4, 256]` 과 `[4, 256]` 으로 다른데 어떻게 더해질까요? PyTorch 의 **브로드캐스팅(broadcasting)** 덕분입니다. 작은 쪽(`[4, 256]`)을 자동으로 8번 복사한 것처럼 취급해서 8개 샘플 모두에 **똑같은 위치 벡터**를 더해 줍니다.

```
 token_embeddings   [8, 4, 256]     what the token is
 pos_embeddings        [4, 256]     where it is         (copied to all 8 samples)
 -----------------------------  +
 input_embeddings   [8, 4, 256]     -> goes into the model
```

### 실행 결과 읽기
세 줄의 shape (`[8, 4, 256]`, `[4, 256]`, `[8, 4, 256]`) 가 위 설명과 일치하면 성공입니다.
(저장된 출력 끝에 "Kernel crashed" 메시지가 남아 있는데, shape 세 줄은 정상 출력된 뒤의 일이라 코드 논리가 틀린 것은 아닙니다. 다시 실행할 때는 셀 1부터 순서대로 실행해 보세요.)

In [ ]:
# 8
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

context_length = max_length
# region [포지셔널 임베딩 레이어 생성]
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
# endregion
# region [포지셔널 임베딩 적용]
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
# endregion
print(pos_embeddings.shape)
# region [토큰 임베딩과 포지셔널 임베딩 합산]
input_embeddings = token_embeddings + pos_embeddings
# endregion
print(input_embeddings.shape)


torch.Size([8, 4, 256])
torch.Size([4, 256])
torch.Size([8, 4, 256])


: 

## Tokenizer 비교

- 아래 결과를 보면 한국어는 Exaone, 중국어는 Qwen의 Tokenizer 성능이 좋습니다

---
## 셀 9 — 토크나이저 비교 (Qwen3 vs EXAONE)

### 🎯 이 셀이 하는 일
지금까지는 GPT-2 토크나이저만 썼습니다. 이번에는 **중국(알리바바)의 Qwen3** 와 **한국(LG)의 EXAONE** 토크나이저를 불러와, 같은 문장을 **몇 개의 토큰으로 쪼개는지** 비교합니다.

**토큰 수가 적을수록 효율적**입니다. 같은 글을 더 적은 토큰으로 표현하면 모델이 한 번에 더 긴 글을 볼 수 있고 계산도 빨라집니다.

### 코드 뜯어보기

**① 토크나이저 불러오기**
```python
from transformers import AutoTokenizer
qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
exaone = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-4.0-1.2B")
```
`transformers` 는 Hugging Face 의 라이브러리입니다. `from_pretrained("모델 이름")` 은 Hugging Face Hub(모델 저장소)에서 그 모델의 토크나이저 파일을 내려받아 준비합니다. (처음 실행 시 인터넷 필요)

**② 테스트 문장 목록**
```python
texts = ["안녕하세요, 반갑습니다!", "Hello, nice to meet you!", ...]
```
한국어·영어·일본어·중국어·혼합·특수문자 등 다양한 문장을 **리스트**에 담았습니다.

**③ 문장마다 반복**
```python
for text in texts:
    qwen_tokens = qwen.encode(text, add_special_tokens=True)
    qwen_token_strs = [qwen.decode([tok]) for tok in qwen_tokens[:10]]
```
- `for text in texts:` : 리스트에서 문장을 하나씩 꺼내 아래 블록을 반복합니다.
- `qwen.encode(text, ...)` : 글 → 토큰 ID 리스트
- 그다음 줄은 **리스트 컴프리헨션**입니다. 풀어 쓰면 다음과 같습니다.

```python
qwen_token_strs = []
for tok in qwen_tokens[:10]:               # 앞에서 10개까지만
    qwen_token_strs.append(qwen.decode([tok]))   # 토큰 1개씩 글자로 되돌림
```
즉 "토큰을 하나씩 따로 디코딩해서, **어떻게 쪼개졌는지** 글자로 보여 주는" 코드입니다. `decode([tok])` 에서 `[tok]` 으로 감싼 것은 decode 가 리스트를 받기 때문입니다.

**④ 비교**
```python
diff = len(qwen_tokens) - len(exaone_tokens)
if diff > 0:   print(f"→ EXAONE이 {diff}개 더 효율적")
elif diff < 0: print(f"→ Qwen3가 {abs(diff)}개 더 효율적")
else:          print(f"→ 동일")
```
토큰 수의 차이를 구합니다. `abs()` 는 절댓값(부호를 뗀 값)입니다.

**⑤ 특수 토큰 확인**
```python
for token_name in ['bos_token', 'eos_token', 'unk_token', 'pad_token']:
    token = getattr(qwen, token_name, None)
    if token:
        token_id = getattr(qwen, f"{token_name}_id", None)
```
- `getattr(객체, "이름", None)` : 객체에서 **문자열로 된 이름**의 속성을 꺼냅니다. `qwen.bos_token` 이라고 쓰는 것과 같지만, 이름을 변수로 바꿔 가며 반복할 수 있습니다. 없으면 `None`.
- `if token:` : 값이 있을 때만(None 이 아닐 때만) 출력합니다.

| 특수 토큰 | 뜻 |
|---|---|
| `bos_token` | Beginning Of Sequence — 문장 시작 표시 |
| `eos_token` | End Of Sequence — 문장 끝 표시 |
| `unk_token` | Unknown — 모르는 글자 대신 쓰는 표시 |
| `pad_token` | Padding — 길이를 맞추려고 채워 넣는 빈칸 표시 |

### 실행 결과 읽기
```
텍스트: 안녕하세요, 반갑습니다!
Qwen3:  8개 토큰   ['안', '녕', '하세요', ',', ' 반', '갑', '습니다', '!']
EXAONE: 7개 토큰   ['안녕', '하', '세요', ',', ' 반갑', '습니다', '!']
```
EXAONE 은 `안녕`, `반갑` 을 한 덩어리로 알고 있지만, Qwen3 는 `안` / `녕` 으로 쪼갭니다. **한국어 글을 많이 보고 만든 토크나이저일수록 한국어를 덜 쪼갠다**는 것을 알 수 있습니다. 반대로 중국어 문장은 Qwen3 쪽이 유리합니다. 영어는 둘이 거의 같습니다.

In [9]:
# 9
from transformers import AutoTokenizer
# Tokenizer 로드 (로컬 또는 온라인에서)
print("Tokenizer 로드 중...")
qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
exaone = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-4.0-1.2B")

# 테스트할 텍스트
texts = [
    "안녕하세요, 반갑습니다!",
    "Hello, nice to meet you!",
    "인공지능 기술이 발전하고 있습니다.",
    "Hello! This is an English tokenizer test.",
    "こんにちは！日本語のテストです。",
    "人工智能技术正在快速发展。",
    "The quick brown fox jumps over the lazy dog. 빠른 갈색 여우가 게으른 개를 뛰어넘습니다.",
    "파이썬 프로그래밍은 매우 유용합니다. Python programming is very useful.",
    "123456789 !@#$%^&*() 특수문자",
]

print("\n" + "="*60)
for text in texts:
    print(f"\n텍스트: {text}")
    print("-"*60)
    
    # Qwen3
    qwen_tokens = qwen.encode(text, add_special_tokens=True)
    qwen_token_strs = [qwen.decode([tok]) for tok in qwen_tokens[:10]]
    print(f"Qwen3:  {len(qwen_tokens)}개 토큰")
    print(f"        {qwen_token_strs}")
    
    # EXAONE
    exaone_tokens = exaone.encode(text, add_special_tokens=True)
    exaone_token_strs = [exaone.decode([tok]) for tok in exaone_tokens[:10]]
    print(f"EXAONE: {len(exaone_tokens)}개 토큰")
    print(f"        {exaone_token_strs}")
    
    # 비교
    diff = len(qwen_tokens) - len(exaone_tokens)
    if diff > 0:
        print(f"→ EXAONE이 {diff}개 더 효율적")
    elif diff < 0:
        print(f"→ Qwen3가 {abs(diff)}개 더 효율적")
    else:
        print(f"→ 동일")

    # Special tokens 비교
print("\n" + "=" * 80)
print("4. Special Tokens 비교")
print("=" * 80)

print(f"\n[Qwen3 Special Tokens]")
for token_name in ['bos_token', 'eos_token', 'unk_token', 'pad_token']:
    token = getattr(qwen, token_name, None)
    if token:
        token_id = getattr(qwen, f"{token_name}_id", None)
        print(f"  {token_name}: '{token}' (ID: {token_id})")

print(f"\n[EXAONE Special Tokens]")
for token_name in ['bos_token', 'eos_token', 'unk_token', 'pad_token']:
    token = getattr(exaone, token_name, None)
    if token:
        token_id = getattr(exaone, f"{token_name}_id", None)
        print(f"  {token_name}: '{token}' (ID: {token_id})")

Tokenizer 로드 중...




텍스트: 안녕하세요, 반갑습니다!
------------------------------------------------------------
Qwen3:  8개 토큰
        ['안', '녕', '하세요', ',', ' 반', '갑', '습니다', '!']
EXAONE: 7개 토큰
        ['안녕', '하', '세요', ',', ' 반갑', '습니다', '!']
→ EXAONE이 1개 더 효율적

텍스트: Hello, nice to meet you!
------------------------------------------------------------
Qwen3:  7개 토큰
        ['Hello', ',', ' nice', ' to', ' meet', ' you', '!']
EXAONE: 7개 토큰
        ['Hello', ',', ' nice', ' to', ' meet', ' you', '!']
→ 동일

텍스트: 인공지능 기술이 발전하고 있습니다.
------------------------------------------------------------
Qwen3:  12개 토큰
        ['인', '공', '지', '능', ' 기', '술', '이', ' 발', '전', '하고']
EXAONE: 9개 토큰
        ['인공', '지능', ' 기술', '이', ' 발전', '하고', ' 있', '습니다', '.']
→ EXAONE이 3개 더 효율적

텍스트: Hello! This is an English tokenizer test.
------------------------------------------------------------
Qwen3:  9개 토큰
        ['Hello', '!', ' This', ' is', ' an', ' English', ' tokenizer', ' test', '.']
EXAONE: 9개 토큰
        ['Hello', '!', ' This', ' is

## Hugging Face 모델 파일 구조 설명

Hugging Face Hub에 업로드된 파일들은 **Transformer 기반 언어 모델(LLM)**을 구성하는 핵심 요소들입니다. 크게 **설계도(Config)**, **가중치(Weights)**, **토크나이저(Tokenizer)** 세 그룹으로 나뉩니다.

---

### 1. 모델 아키텍처 및 설정 (설계도)

* **`config.json`**
    * **의미:** 모델의 **구조와 설계도**가 담긴 가장 중요한 설정 파일입니다.
    * **내용:** 모델의 종류(Llama, BERT 등), 레이어(Layer) 수, 히든 사이즈, 어텐션 헤드 수 등을 정의하여 빈 모델(Skeleton)을 만듭니다.

* **`generation_config.json`**
    * **의미:** 텍스트 **생성(Generation) 시 사용할 기본 옵션** 파일입니다.
    * **내용:** `max_length`(최대 길이), `temperature`(창의성), `top_p`, `eos_token_id`(종료 토큰) 등의 기본 설정값이 들어있습니다.

### 2. 모델 가중치 (학습된 뇌)

* **`model.safetensors`**
    * **의미:** 실제 **학습된 파라미터(가중치)**가 저장된 파일입니다. (가장 용량이 큼)
    * **특징:**
        * 기존 PyTorch의 `.bin` 파일보다 로딩 속도가 빠릅니다.
        * 보안 취약점(악성 코드 실행 등)을 해결한 안전한 형식입니다.

### 3. 토크나이저 (언어 처리기)
사람의 언어(Text)를 모델이 이해하는 숫자(Token ID)로 변환하는 규칙들입니다.

* **`vocab.json`**
    * **의미:** **단어장(Vocabulary)** 파일입니다. 단어와 그에 대응하는 숫자 ID가 매핑되어 있습니다.
* **`merges.txt`**
    * **의미:** **BPE 병합 규칙** 파일입니다. 알파벳 조각들이 어떻게 합쳐져서 단어가 되는지 정의합니다.
* **`tokenizer.json`**
    * **의미:** `vocab.json`과 `merges.txt` 등을 하나로 통합하여 최적화한 파일로, 로딩 속도가 매우 빠릅니다.
* **`tokenizer_config.json`**
    * **의미:** 토크나이저 설정 파일입니다. 어떤 클래스를 쓸지, 특수 토큰(`<s>`, `<pad>` 등)은 무엇인지 정의합니다.

---

### 요약

| 파일명 | 비유 | 역할 |
| :--- | :--- | :--- |
| `config.json` | **설계도** | 모델 구조 정의 |
| `model.safetensors` | **뇌** | 학습된 지식(가중치) 저장 |
| `tokenizer.json` | **번역기** | 텍스트 ↔ 숫자 변환 |
| `generation_config.json` | **말하기 습관** | 생성 옵션 설정 |

## Simulation
https://tiktokenizer.vercel.app/?model=cl100k_base


---
## 📌 Chapter 2 정리

```
 raw text
    |  tokenizer.encode            (cell 2, 3)
    v
 token IDs  [40, 367, 2885, ...]
    |  sliding window              (cell 4 : GPTDatasetV1)
    v
 (input, target) pairs             target = input shifted by 1
    |  DataLoader                  (cell 5, 6)
    v
 batch  [batch_size, max_length]
    |  token embedding + position embedding   (cell 7, 8)
    v
 input_embeddings  [batch_size, max_length, emb_dim]   ->  Chapter 3 (Attention)
```

**꼭 기억할 3가지**
1. 모델은 글자가 아니라 **토큰 ID** 를 본다.
2. 정답(target)은 입력(input)을 **한 칸 민 것** — 사람이 정답을 달 필요가 없다.
3. 모델 입력 = **토큰 임베딩(무엇인가) + 위치 임베딩(어디에 있는가)**

다음 챕터(3장)에서는 이렇게 만든 벡터들이 서로를 "참고"하는 방법, 즉 **어텐션**을 만듭니다.